# Homework 11: Bootstrap, CLT, and Normal Approximation

Problems 1-6 are worked out below. For problems 7-10, I ran the code with the CSV files to get the actual numbers.

Formulas I used:

SE for one mean = s / sqrt(n)
SE for two means = sqrt(sA^2/nA + sB^2/nB)
z = (estimate - null value) / SE
CI = estimate +/- critical value * SE

Critical values: 90% = 1.645, 95% = 1.96, 99% = 2.576.

Problem 1

SE = 10 / sqrt(100) = 1
z = (52 - 50) / 1 = 2

At 0.10, 2 > 1.645 so reject H0.
At 0.05, 2 > 1.96 so reject H0.
At 0.01, 2 < 2.576 so fail to reject H0.

95% CI = 52 +/- 1.96(1) = (50.04, 53.96)

There is evidence the mean is different from 50 at the 10% and 5% levels, but not at the 1% level.

Problem 2

z = 3.1 / 1.3 = 2.385

90% CI = 3.1 +/- 1.645(1.3) = (0.9615, 5.2385)
95% CI = 3.1 +/- 1.96(1.3) = (0.552, 5.648)
99% CI = 3.1 +/- 2.576(1.3) = (-0.2488, 6.4488)

Reject H0 at 0.10 and 0.05 since 0 is not in those intervals. Fail to reject at 0.01 since 0 is in the 99% interval.

Problem 3

SE = 3 / sqrt(36) = 0.5
z = (11.5 - 12) / 0.5 = -1
95% CI = 11.5 +/- 1.96(0.5) = (10.52, 12.48)

Fail to reject H0 at 0.05 since 12 is in the interval and the absolute z value is less than 1.96. So 12 is a plausible mean.

Problem 4

Difference = 25 - 22 = 3
SE = sqrt(8^2/100 + 7^2/100) = sqrt(1.13) = 1.063
z = 3 / 1.063 = 2.822
95% CI = 3 +/- 1.96(1.063) = (0.917, 5.083)

Reject H0 at 0.05 because the interval does not include 0. Neighborhood A has a higher sample mean, and there is evidence the population means differ.

Problem 5

Difference = 14 - 12 = 2
SE = sqrt(6^2/81 + 5^2/64) = sqrt(0.8351) = 0.914
z = 2 / 0.914 = 2.188

Since 2.188 > 1.96, reject H0 at 0.05. There is evidence the average recovery times differ, with treatment A taking longer on average.

Problem 6

Difference = 3.2 - 2.8 = 0.4
SE = sqrt(1.5^2/49 + 1.4^2/64) = 0.277
z = 0.4 / 0.277 = 1.446
95% CI = 0.4 +/- 1.96(0.277) = (-0.142, 0.942)

Fail to reject H0 at 0.05 since 1.446 < 1.96 and the interval includes 0. The difference is not statistically significant.

Problems 7-10

The code below uses 5000 bootstrap samples and seed 42. I need the CSV files to get the results. I used the bootstrap standard deviation with ddof=1.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

CRITICAL = {90: 1.645, 95: 1.96, 99: 2.576}
SEED = 42
B = 5000

def load_csv(filename):
    for path in [Path("data") / filename, Path(filename), Path("/content") / filename, Path("/content/data") / filename]:
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(f"Upload {filename} to Colab or place it in ./data/")

def report_intervals(estimate, se, null, unit=""):
    print(f"Estimate: {estimate:.6f} {unit}; bootstrap SE: {se:.6f}")
    for level, z in CRITICAL.items():
        lo, hi = estimate - z * se, estimate + z * se
        decision = "REJECT" if not (lo <= null <= hi) else "FAIL TO REJECT"
        print(f"{level}% CI: ({lo:.6f}, {hi:.6f}) | H0={null}: {decision} at alpha={1-level/100:.2f}")


Problem 7

a. Find the proportion of homes with Central.Air equal to Y.
b. Bootstrap the proportion using seed 42 and 5000 samples.
c. The bootstrap SE is the standard deviation of the bootstrap estimates.
d. Use estimate +/- critical value * SE for the 90%, 95%, and 99% intervals.
e. For each interval, reject H0: p = 0.90 if 0.90 is outside the interval. Otherwise fail to reject.

The code prints the numbers after loading the data.

In [2]:
ames = load_csv("ames_prices.csv")
central_air = (ames["Central.Air"] == "Y").astype(int).to_numpy()
p_hat = central_air.mean()
rng = np.random.default_rng(SEED)
boot_p = np.array([rng.choice(central_air, size=len(central_air), replace=True).mean() for _ in range(B)])
se_p = np.std(boot_p, ddof=1)
print("Seed:", SEED, "Bootstrap replications:", B)
report_intervals(p_hat, se_p, null=0.90)


Seed: 42 Bootstrap replications: 5000
Estimate: 0.933106 ; bootstrap SE: 0.004692
90% CI: (0.925387, 0.940825) | H0=0.9: REJECT at alpha=0.10
95% CI: (0.923909, 0.942303) | H0=0.9: REJECT at alpha=0.05
99% CI: (0.921018, 0.945194) | H0=0.9: REJECT at alpha=0.01


a. The estimated proportion of homes with central air conditioning is 0.933106, or about 93.31%.

b. I used a random seed of 42 and 5000 bootstrap replications to estimate the proportion.

c. The bootstrap standard error is 0.004692.

d. Using the original estimate and bootstrap standard error, I calculated these confidence intervals:

90% CI: (0.925387, 0.940825)

95% CI: (0.923909, 0.942303)

99% CI: (0.921018, 0.945194)

e. I reject the null hypothesis that p = 0.90 at all three significance levels (0.10, 0.05, and 0.01). This is because 0.90 is outside all three confidence intervals. The results suggest that the proportion of homes with central air is greater than 90%.

Problem 8

First calculate the Pearson correlation between arcsinh(area) and arcsinh(price). Then bootstrap 5000 times with seed 42, resampling whole rows so area and price stay together. Use the standard deviation of the correlations for SE and calculate the 90%, 95%, and 99% intervals. Reject H0: rho = 0 when the interval does not contain 0.

Area and price have to stay paired because each pair belongs to the same house. If they are sampled separately, it breaks the relationship we are trying to measure. Correlation does not prove causation.

In [3]:
pairs = load_csv("ames_prices.csv")[["area", "price"]].dropna()
x = np.arcsinh(pairs["area"].to_numpy())
y = np.arcsinh(pairs["price"].to_numpy())
r_hat = np.corrcoef(x, y)[0, 1]
rng = np.random.default_rng(SEED)
boot_r = []
for _ in range(B):
    indices = rng.choice(len(x), size=len(x), replace=True)
    boot_r.append(np.corrcoef(x[indices], y[indices])[0, 1])
se_r = np.std(boot_r, ddof=1)
print("Seed:", SEED, "Bootstrap replications:", B)
report_intervals(r_hat, se_r, null=0)


Seed: 42 Bootstrap replications: 5000
Estimate: 0.723166 ; bootstrap SE: 0.010224
90% CI: (0.706348, 0.739984) | H0=0: REJECT at alpha=0.10
95% CI: (0.703127, 0.743205) | H0=0: REJECT at alpha=0.05
99% CI: (0.696829, 0.749503) | H0=0: REJECT at alpha=0.01


The estimated Pearson correlation between the transformed area and price is 0.723166.

a. I used a random seed of 42 and 5000 bootstrap replications, resampling the paired observations.

b. The bootstrap standard error is 0.010224.

c. The confidence intervals are:

90% CI: (0.706348, 0.739984)

95% CI: (0.703127, 0.743205)

99% CI: (0.696829, 0.749503)

d. I reject the null hypothesis that the correlation is zero at all three significance levels. None of the confidence intervals contain zero. This suggests a strong positive relationship between transformed area and price, meaning larger homes tend to have higher prices. However, this does not prove causation.

e. Area and price need to stay paired during resampling because they belong to the same house. If they were resampled separately, the original relationship between area and price would be lost, making the correlation estimate unreliable.

Problem 9

Use Overall Survival (Months) for all patients, including living patients. Calculate the mean and bootstrap it 5000 times with seed 42. Use the bootstrap SE for the 90%, 95%, and 99% intervals. Reject H0: mean = 120 months if 120 is outside the interval.

To convert the estimate and intervals to years, divide by 12. This is average recorded time to death or last follow-up, not average lifetime, since living patients have not necessarily reached their eventual death time.

In [4]:
metabric = load_csv("metabric.csv")
months = pd.to_numeric(metabric["Overall Survival (Months)"], errors="coerce").dropna().to_numpy()
mean_months = months.mean()
rng = np.random.default_rng(SEED)
boot_months = np.array([rng.choice(months, size=len(months), replace=True).mean() for _ in range(B)])
se_months = np.std(boot_months, ddof=1)
print("Seed:", SEED, "Bootstrap replications:", B)
report_intervals(mean_months, se_months, null=120, unit="months")
print(f"Mean years: {mean_months / 12:.4f}")
for level, z in CRITICAL.items():
    print(f"{level}% CI in years: ({(mean_months-z*se_months)/12:.4f}, {(mean_months+z*se_months)/12:.4f})")


Seed: 42 Bootstrap replications: 5000
Estimate: 129.094291 months; bootstrap SE: 2.142386
90% CI: (125.570066, 132.618516) | H0=120: REJECT at alpha=0.10
95% CI: (124.895215, 133.293368) | H0=120: REJECT at alpha=0.05
99% CI: (123.575505, 134.613078) | H0=120: REJECT at alpha=0.01
Mean years: 10.7579
90% CI in years: (10.4642, 11.0515)
95% CI in years: (10.4079, 11.1078)
99% CI in years: (10.2980, 11.2178)


The estimated mean recorded survival or follow-up time is 129.094291 months, or about 10.76 years.

a. I used a random seed of 42 and 5000 bootstrap replications to estimate the mean.

b. The bootstrap standard error is 2.142386 months.

c. The confidence intervals are:

90% CI: (125.570066, 132.618516) months

95% CI: (124.895215, 133.293368) months

99% CI: (123.575505, 134.613078) months

d. I reject the null hypothesis that the mean is 120 months at all three significance levels. All three confidence intervals are above 120 months, suggesting that the mean recorded survival or follow-up time is greater than 120 months.

e. The average recorded time to death or last follow-up is about 10.76 years.

90% CI: (10.4642, 11.0515) years

95% CI: (10.4079, 11.1078) years

99% CI: (10.2980, 11.2178) years

These results include both living and deceased patients. Since living patients have not necessarily reached their eventual death time, their data are right-censored. This means the results estimate average recorded survival or follow-up time, not the actual average lifetime of patients.

Problem 10

Only include people age 18 or older who have a weekday sleep value. Calculate the mean, then bootstrap it 5000 times with seed 42. Use the bootstrap SE to make the 90%, 95%, and 99% intervals. Reject H0: mean = 8 hours when 8 is outside the interval.

Even if the difference from 8 hours is statistically significant, it might be too small to matter much in practice. This analysis ignores survey weights and the survey design, so I cannot call it nationally representative.

In [5]:
nhanes = load_csv("nhanes_data_17_18.csv")
age_candidates = ["Age", "age", "RIDAGEYR", "AgeInYearsAtScreening"]
age_col = next((c for c in age_candidates if c in nhanes.columns), None)
if age_col is None:
    raise KeyError(f"Age column not identified. Inspect columns and set age_col manually: {nhanes.columns.tolist()}")
sleep_col = "SleepHoursWeekdaysOrWorkdays"
selected = nhanes.loc[pd.to_numeric(nhanes[age_col], errors="coerce") >= 18, sleep_col]
sleep = pd.to_numeric(selected, errors="coerce").dropna().to_numpy()
mean_sleep = sleep.mean()
rng = np.random.default_rng(SEED)
boot_sleep = np.array([rng.choice(sleep, size=len(sleep), replace=True).mean() for _ in range(B)])
se_sleep = np.std(boot_sleep, ddof=1)
print("Age column:", age_col, "Adult observations:", len(sleep))
print("Seed:", SEED, "Bootstrap replications:", B)
report_intervals(mean_sleep, se_sleep, null=8, unit="hours")
print(f"Difference from 8 hours: {mean_sleep-8:.4f} hours = {(mean_sleep-8)*60:.2f} minutes")


/var/folders/8l/3fv02m2d4z7fzxzg1_szh51h0000gn/T/ipykernel_7755/337124636.py:12: DtypeWarning: Columns (0: ServedActiveDutyInUsArmedForces) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)


Age column: AgeInYearsAtScreening Adult observations: 5489
Seed: 42 Bootstrap replications: 5000
Estimate: 7.621242 hours; bootstrap SE: 0.022587
90% CI: (7.584087, 7.658398) | H0=8: REJECT at alpha=0.10
95% CI: (7.576972, 7.665513) | H0=8: REJECT at alpha=0.05
99% CI: (7.563058, 7.679427) | H0=8: REJECT at alpha=0.01
Difference from 8 hours: -0.3788 hours = -22.73 minutes


The estimated mean weekday sleep for adults is 7.621242 hours, based on 5489 observations.

a. I used a random seed of 42 and 5000 bootstrap replications to estimate the mean weekday sleep.

b. The bootstrap standard error is 0.022587 hours.

c. The confidence intervals are:

90% CI: (7.584087, 7.658398) hours

95% CI: (7.576972, 7.665513) hours

99% CI: (7.563058, 7.679427) hours

d. I reject the null hypothesis that the mean weekday sleep is 8 hours at all three significance levels. This is because none of the confidence intervals contain 8. The results suggest that the mean weekday sleep is less than 8 hours.

e. The estimated average weekday sleep is about 7.62 hours, which is approximately 22.73 minutes less than 8 hours. This difference is statistically significant because all three confidence intervals exclude 8 hours. It could also be practically meaningful since losing around 23 minutes of sleep each night may add up over time. However, statistical significance does not necessarily mean the difference is important for everyone. Since the analysis ignores survey weights and the complex sampling design, the results should not be considered nationally representative.